<a href="https://colab.research.google.com/github/visista19/30-days-challenge/blob/main/Customeraiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.7/160.7 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.5/810.5 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.2/342.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 23.5 M

In [ ]:
from crewai import Agent, Task, Crew

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [ ]:
import os
from google.colab import userdata

# 1. Fetch your Groq API key from Colab Secrets
groq_key = userdata.get('GROK_API_KEY')

# 2. Redirect CrewAI's default OpenAI routing to Groq's infrastructure
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["OPENAI_API_KEY"] = groq_key
os.environ["OPENAI_MODEL_NAME"] = "llama-3.3-70b-versatile"

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	verbose=True
)

In [ ]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	verbose=True
)

In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

In [ ]:
# Possible Custom Tools
# Load customer data
# Tap into previous conversations
# Load data from a CRM
# Checking existing bug reports
# Checking existing feature requests
# Checking ongoing tickets
# ... and more

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)

In [ ]:
# Different Ways to Give Agents Tools
# Agent Level: The Agent can use the Tool(s) on any Task it performs.
# Task Level: The Agent will only use the Tool(s) when performing that specific Task.
# Note: Task Tools override the Agent Tools.

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


In [ ]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=2,
  memory=False
)

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Senior Support Representative
 [INFO]: == Starting Task: DeepLearningAI just reached out with a super important ask:
I need help with setting up a Crew and kicking it off, specifically how can I add memory to my crew? Can you provide guidance?

Andrew Ng from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide the best support possible.You must strive to provide a complete and accurate response to the customer's inquiry.


> Entering new CrewAgentExecutor chain...
Thought: To provide the best support to Andrew Ng from DeepLearningAI, I need to understand how to set up a Crew and kick it off, specifically how to add memory to the Crew. The first step is to gather information from the available resources.

Action: Read website content
Action Input: {"url": "https://docs.crewai.com/en/enterprise/guides/kickoff-crew"} 

Kickoff Crew - CrewAIDocumentation IndexFetch the complete documentation index at: /llms.txtUse this fil

In [ ]:
from IPython.display import Markdown
Markdown(result)

To address your inquiry about setting up a Crew and adding memory to it, I have compiled a comprehensive response that covers all aspects of your question. 

First, setting up a Crew involves several key steps that are crucial for its effective operation and integration with the existing infrastructure. You should start by logging into your crewAI account and navigating to the Crew management section. Here, you can find an option to create a new Crew or edit an existing one, depending on your specific needs. It is essential to carefully follow the on-screen instructions during this process, as each step is designed to ensure that the Crew is properly configured for optimal performance.

Once the Crew is set up, the next step is to add memory to it, which is a critical component for enhancing its capabilities and handling more complex tasks. The process of adding memory typically involves accessing the Crew's settings or configuration panel, where you can find options related to memory allocation. The exact steps may vary depending on the specific crewAI platform version or the type of Crew being utilized, so it is vital to consult the most recent documentation or support resources provided by crewAI for the most accurate and up-to-date instructions.

Here is a general, step-by-step guide that you can follow to add memory to a Crew:

1. **Log in to your crewAI account**: Start by accessing your crewAI account through the official website or application. Ensure you are using the correct credentials to avoid any login issues.

2. **Navigate to the Crew management section**: After logging in, locate the section dedicated to managing Crews. This is usually found in a dashboard or control panel where all your Crews are listed.

3. **Select the Crew to modify**: Identify the specific Crew you want to add memory to and select it from the list. This action should open a detailed view or settings page for that Crew.

4. **Access the memory settings**: Within the Crew's settings, look for an option related to memory or resource allocation. This could be under a tab named "Resources," "Settings," or "Configuration," depending on the interface.

5. **Adjust the memory allocation**: Once you have found the memory settings, you should see an option to increase the memory allocated to the Crew. This might be in the form of a slider, dropdown menu, or input field where you can enter the desired memory amount.

6. **Confirm the changes**: After adjusting the memory allocation, make sure to save or confirm your changes. The system may prompt you to verify that you want to proceed with the modification, especially if it involves additional costs or significant changes to your Crew's configuration.

7. **Monitor the Crew's performance**: After adding memory, it is a good practice to monitor the Crew's performance to ensure that the changes have the desired effect. This can involve checking for improvements in task execution speed, increased capacity to handle complex tasks, or other performance metrics relevant to your use case.

In addition to these steps, it is crucial for you to be aware of any potential limitations or considerations when adding memory to a Crew. These might include:

- **Cost implications**: Increasing the memory of a Crew may incur additional costs, depending on the crewAI pricing model and your current plan.
- **Technical requirements**: Ensure that the hardware or infrastructure supporting the Crew can handle the increased memory demand without compromising performance or stability.
- **Optimization**: Simply adding more memory might not always result in better performance. You should consider optimizing your Crew's configuration and the tasks it handles for the best outcome.

To ensure that all aspects of your inquiry have been thoroughly addressed, I recommend that you also review the official crewAI documentation and support resources. These materials can provide detailed guides, troubleshooting tips, and insights into best practices for setting up and optimizing Crews within the crewAI platform.

By following these steps and considering the factors mentioned, you should be able to successfully set up a Crew and add memory to it, thereby enhancing its capabilities and performance. If any issues arise during this process or if you have further questions, you are encouraged to reach out to the crewAI support team for personalized assistance.

References:
- CrewAI Documentation: https://docs.crewai.com/en/enterprise/guides/kickoff-crew
- CrewAI Support Team: The support team can be contacted for assistance with execution issues or questions about the Enterprise platform.

I hope this information is helpful. If you have any further questions, please do not hesitate to ask. Your satisfaction is our top priority, and I'm committed to providing you with the best possible support experience.